# ViFinQA CCL Phase 3 GPU Bake-off V1

This kernel runs one hash-bound text route at a time, beginning with **Qwen3-8B in 4-bit**. It validates immutable source/job inputs before model download, records raw model text, then validates source anchors.

It never certifies a fact, edits a source table, creates a training label, or promotes a model. A valid response remains `VALID_PROPOSAL_ONLY`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile

WORKING = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
SOURCE_DIR = WORKING / 'ai_guru_ccl_phase3_source_v1'
QWEN_RAW_DIR = WORKING / 'ccl_qwen3_8b_raw_v1'
QWEN_VALIDATED_DIR = WORKING / 'ccl_qwen3_8b_validated_v1'

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for CCL Phase 3. Enable a Kaggle GPU accelerator.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
if GPU_VRAM_GIB < 12:
    raise RuntimeError(f'CCL Qwen3-8B 4-bit route requires at least 12 GiB VRAM; observed {GPU_VRAM_GIB} GiB.')
print({'gpu': GPU_NAME, 'vram_gib': GPU_VRAM_GIB, 'python': sys.version.split()[0]})

## Required Kaggle inputs

Attach two private datasets before running:

1. `dungle2810/vifinqa-ccl-phase3-source-v1`: the `.bundle` and its source manifest;
2. `dungle2810/vifinqa-ccl-phase3-bakeoff-v1`: the job manifest, requests and response schema.

Enable Internet for Hugging Face model downloads. Gemma is optional because its model-card terms/access must be accepted separately.

In [ ]:
SOURCE_ARCHIVE_NAME = 'ai_guru_ccl_phase3_source_v1.bundle'
SOURCE_MANIFEST_NAME = 'ai_guru_ccl_phase3_source_v1.manifest.json'
JOB_MANIFEST_NAME = 'bakeoff_job_manifest.json'
REQUESTS_NAME = 'llm_bakeoff_requests_v1.jsonl'
RESPONSE_SCHEMA_NAME = 'llm_proposal_schema_v1.json'

def exactly_one_input(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one Kaggle input named {name!r}; found {matches}')
    return matches[0]

SOURCE_ARCHIVE = exactly_one_input(SOURCE_ARCHIVE_NAME)
SOURCE_MANIFEST = exactly_one_input(SOURCE_MANIFEST_NAME)
JOB_MANIFEST = exactly_one_input(JOB_MANIFEST_NAME)
REQUESTS = exactly_one_input(REQUESTS_NAME)
RESPONSE_SCHEMA = exactly_one_input(RESPONSE_SCHEMA_NAME)
print({
    'source_archive': str(SOURCE_ARCHIVE),
    'job_manifest': str(JOB_MANIFEST),
    'request_count': sum(1 for line in REQUESTS.open(encoding='utf-8') if line.strip()),
})

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(value: object) -> str:
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')).hexdigest()

if SOURCE_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite extracted source directory: {SOURCE_DIR}')
source_manifest = json.loads(SOURCE_MANIFEST.read_text(encoding='utf-8'))
if source_manifest.get('protocol') != 'kaggle_ccl_phase3_source_bundle_v1':
    raise RuntimeError('Unexpected CCL source-bundle protocol.')
archive_contract = (source_manifest.get('outputs') or {}).get('archive') or {}
if archive_contract.get('file_name') != SOURCE_ARCHIVE.name or archive_contract.get('sha256') != sha256_file(SOURCE_ARCHIVE):
    raise RuntimeError('Source bundle archive does not match its manifest SHA-256.')
for key, value in (source_manifest.get('source_contract') or {}).items():
    if value is not False:
        raise RuntimeError(f'Source isolation contract violated: {key}={value!r}')
identity = source_manifest.get('source_bundle') or {}
identity_without_tree = {key: value for key, value in identity.items() if key != 'source_tree_sha256'}
if identity.get('source_tree_sha256') != canonical_sha256(identity_without_tree):
    raise RuntimeError('Source-tree identity hash is invalid.')
files = identity.get('files') or []
expected_names = [entry['path'] for entry in files] + ['SOURCE_BUNDLE.json']
SOURCE_DIR.mkdir(parents=True)
with tarfile.open(SOURCE_ARCHIVE, mode='r:*') as archive:
    members = archive.getmembers()
    if [member.name for member in members] != expected_names or not all(member.isfile() for member in members):
        raise RuntimeError('Source archive members do not exactly match its manifest identity.')
    embedded = archive.extractfile('SOURCE_BUNDLE.json')
    if embedded is None or json.loads(embedded.read()) != identity:
        raise RuntimeError('Embedded source identity differs from source manifest.')
    for entry in files:
        name, expected_sha = entry['path'], entry['sha256']
        if Path(name).is_absolute() or '..' in Path(name).parts:
            raise RuntimeError(f'Unsafe source member path: {name}')
        handle = archive.extractfile(name)
        if handle is None or hashlib.sha256(handle.read()).hexdigest() != expected_sha:
            raise RuntimeError(f'Source member checksum mismatch: {name}')
        handle = archive.extractfile(name)
        destination = SOURCE_DIR / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(handle.read())

job_manifest = json.loads(JOB_MANIFEST.read_text(encoding='utf-8'))
if (job_manifest.get('protocol') != 'vifinqa_ccl_phase3_bakeoff_v1'
        or job_manifest.get('run_status') != 'prepared_phase_3_inference_not_executed'
        or job_manifest.get('training_eligible') is not False
        or job_manifest.get('model_execution_recorded') is not False):
    raise RuntimeError('Bake-off job is not a non-promotable prepared Phase 3 job.')
for path in (REQUESTS, RESPONSE_SCHEMA):
    expected_sha = ((job_manifest.get('outputs') or {}).get(path.name) or {}).get('sha256')
    if expected_sha != sha256_file(path):
        raise RuntimeError(f'Job output checksum mismatch: {path.name}')
print({'source_tree_sha256': identity['source_tree_sha256'], 'job_request_sha256': sha256_file(REQUESTS)})

In [ ]:
# Resolve a model runtime compatible with the scheduled accelerator before checkpoint loading.
# Kaggle's current P100 image can ship a PyTorch wheel without SM60 kernels.
GPU_CAPABILITY = tuple(int(value) for value in torch.cuda.get_device_capability(0))
if GPU_CAPABILITY < (6, 0):
    raise RuntimeError(f'NF4 4-bit execution requires compute capability >= 6.0; observed {GPU_CAPABILITY}.')
NEEDS_PASCAL_COMPAT_RUNTIME = GPU_CAPABILITY < (7, 0)
RUNTIME_LOCK = {
    'torch': 'torch==2.6.0',
    'torchvision': 'torchvision==0.21.0',
    'torch_index_url': 'https://download.pytorch.org/whl/cu118',
    'tokenizers': 'tokenizers==0.21.4',
    'huggingface_hub': 'huggingface-hub==0.30.2',
    'safetensors': 'safetensors==0.5.3',
    'transformers': 'transformers==4.51.3',
    'accelerate': 'accelerate==1.7.0',
    'bitsandbytes': 'bitsandbytes==0.45.5',
    'sentencepiece': 'sentencepiece==0.2.0',
}
if NEEDS_PASCAL_COMPAT_RUNTIME:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall',
        '--index-url', RUNTIME_LOCK['torch_index_url'], RUNTIME_LOCK['torch'],
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall', '--no-deps',
        '--index-url', RUNTIME_LOCK['torch_index_url'], RUNTIME_LOCK['torchvision'],
    ], check=True)
# These pins must not resolve dependencies again: PyPI would replace the CUDA-11.8
# wheel above with a newer wheel that has no P100 SM60 kernel.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall', '--no-deps',
    RUNTIME_LOCK['tokenizers'], RUNTIME_LOCK['huggingface_hub'], RUNTIME_LOCK['safetensors'],
    RUNTIME_LOCK['transformers'], RUNTIME_LOCK['accelerate'], RUNTIME_LOCK['bitsandbytes'], RUNTIME_LOCK['sentencepiece'],
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE_DIR), '--no-deps'], check=True)
RUNTIME_PROBE = """
import json
import torch
import torchvision
if not torch.cuda.is_available():
    raise RuntimeError('CUDA disappeared after runtime resolution.')
capability = tuple(int(value) for value in torch.cuda.get_device_capability(0))
arch_list = list(torch.cuda.get_arch_list())
expected_arch = f'sm_{capability[0]}{capability[1]}'
if capability < (6, 0) or expected_arch not in arch_list:
    raise RuntimeError(f'Installed PyTorch cannot run NF4 on {expected_arch}: {arch_list}')
if capability < (7, 0) and not torch.__version__.startswith('2.6.0+cu118'):
    raise RuntimeError(f'P100 PyTorch runtime was overwritten: {torch.__version__}')
if capability < (7, 0) and not torchvision.__version__.startswith('0.21.0+cu118'):
    raise RuntimeError(f'P100 torchvision runtime is mismatched: {torchvision.__version__}')
torch.empty((1,), device='cuda').zero_()
import bitsandbytes
print(json.dumps({
    'cuda_available': True,
    'torch_version': torch.__version__,
    'torchvision_version': torchvision.__version__,
    'torch_cuda_version': torch.version.cuda,
    'gpu_name': torch.cuda.get_device_name(0),
    'gpu_compute_capability': list(capability),
    'torch_arch_list': arch_list,
    'bitsandbytes_version': bitsandbytes.__version__,
}))
"""
probe = subprocess.run([sys.executable, '-c', RUNTIME_PROBE], capture_output=True, text=True)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr, file=sys.stderr)
    raise RuntimeError(f'CCL model runtime preflight failed with exit code {probe.returncode}.')
MODEL_RUNTIME = json.loads(probe.stdout.strip().splitlines()[-1])
print({'runtime_lock': RUNTIME_LOCK, 'model_runtime': MODEL_RUNTIME, 'source_dir': str(SOURCE_DIR)})

## Primary route: Qwen3-8B

This route processes all 270 fixed packets in deterministic decoding mode. The next command produces raw, untrusted model text; the validator then converts invalid/missing responses to `INVALID_UNRESOLVED`.

In [ ]:
subprocess.run([
    sys.executable, str(SOURCE_DIR / 'scripts/run_ccl_phase3_model.py'),
    '--job-manifest', str(JOB_MANIFEST),
    '--requests', str(REQUESTS),
    '--response-schema', str(RESPONSE_SCHEMA),
    '--route-id', 'qwen3_8b_primary',
    '--max-new-tokens', '768',
    '--output-dir', str(QWEN_RAW_DIR),
], check=True)
subprocess.run([
    sys.executable, str(SOURCE_DIR / 'scripts/validate_ccl_phase3_responses.py'),
    '--job-manifest', str(JOB_MANIFEST),
    '--requests', str(REQUESTS),
    '--route-id', 'qwen3_8b_primary',
    '--raw-responses', str(QWEN_RAW_DIR / 'llm_raw_responses_v1.jsonl'),
    '--output-dir', str(QWEN_VALIDATED_DIR),
], check=True)

## Optional independent challenger: Gemma 3 12B

Set `RUN_GEMMA = True` only after the operator has accepted Gemma's access terms and configured an `HF_TOKEN` Kaggle secret if Hugging Face requires one. Its 225 requests cover only nontrivial strata. Failure to access this model is a route availability failure, not a reason to silently substitute another model.

In [ ]:
RUN_GEMMA = False
GEMMA_RAW_DIR = WORKING / 'ccl_gemma3_12b_raw_v1'
GEMMA_VALIDATED_DIR = WORKING / 'ccl_gemma3_12b_validated_v1'

if RUN_GEMMA:
    if not os.environ.get('HF_TOKEN'):
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
            if token:
                os.environ['HF_TOKEN'] = token
        except Exception as error:
            raise RuntimeError('Gemma route requires an accepted model-access path and HF_TOKEN when gated.') from error
    if not os.environ.get('HF_TOKEN'):
        raise RuntimeError('Gemma route not started: no HF_TOKEN is available.')
    subprocess.run([
        sys.executable, str(SOURCE_DIR / 'scripts/run_ccl_phase3_model.py'),
        '--job-manifest', str(JOB_MANIFEST), '--requests', str(REQUESTS),
        '--response-schema', str(RESPONSE_SCHEMA), '--route-id', 'gemma3_12b_challenger',
        '--max-new-tokens', '1024', '--output-dir', str(GEMMA_RAW_DIR),
    ], check=True)
    subprocess.run([
        sys.executable, str(SOURCE_DIR / 'scripts/validate_ccl_phase3_responses.py'),
        '--job-manifest', str(JOB_MANIFEST), '--requests', str(REQUESTS),
        '--route-id', 'gemma3_12b_challenger',
        '--raw-responses', str(GEMMA_RAW_DIR / 'llm_raw_responses_v1.jsonl'),
        '--output-dir', str(GEMMA_VALIDATED_DIR),
    ], check=True)
else:
    print('Gemma challenger intentionally not run in this kernel.')

In [ ]:
qwen_execution = json.loads((QWEN_RAW_DIR / 'model_execution_manifest.json').read_text(encoding='utf-8'))
qwen_validation = json.loads((QWEN_VALIDATED_DIR / 'proposal_validation_manifest.json').read_text(encoding='utf-8'))
for manifest in (qwen_execution, qwen_validation):
    if manifest.get('training_eligible') is not False or manifest.get('certification_allowed') is not False:
        raise RuntimeError('GPU receipt violates non-promotable CCL contract.')
if qwen_execution.get('runtime') != MODEL_RUNTIME:
    raise RuntimeError('Execution runtime differs from the preflight runtime receipt.')
receipt = {
    'schema_version': 1,
    'protocol': 'vifinqa_ccl_phase3_kaggle_kernel_v1',
    'gpu': {'name': GPU_NAME, 'vram_gib': GPU_VRAM_GIB},
    'model_runtime': MODEL_RUNTIME,
    'source_tree_sha256': identity['source_tree_sha256'],
    'job_manifest_sha256': sha256_file(JOB_MANIFEST),
    'qwen_execution_manifest_sha256': sha256_file(QWEN_RAW_DIR / 'model_execution_manifest.json'),
    'qwen_validation_manifest_sha256': sha256_file(QWEN_VALIDATED_DIR / 'proposal_validation_manifest.json'),
    'gemma_requested': RUN_GEMMA,
    'training_eligible': False,
    'certification_allowed': False,
}
(WORKING / 'ccl_phase3_kaggle_receipt_v1.json').write_text(json.dumps(receipt, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(receipt, ensure_ascii=False, indent=2))
print('Download the raw and validated Qwen directories plus ccl_phase3_kaggle_receipt_v1.json.')